In [1]:
import torch
from dinosaw.helpers import ModelTypes, model_names, get_models, get_features
from dinosaw.utils import do_2D_pca

import numpy as np
from tifffile import imread
from PIL import Image

from skimage.transform import resize

import matplotlib.pyplot as plt
from typing import TypeAlias, Literal

from interactive_seg_backend import TrainingConfig, FeatureConfig, featurise, concat_feats, train_and_apply
from interactive_seg_backend.file_handling import load_labels

SEED = 100001
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda:0'
half = False

flash attention installed
N CPUS: 110


In [2]:
enabled_models: tuple[ModelTypes, ...] = ('dv2', 'alibi_dv2_coco', 'dv3', 'alibi_dv3')
models = get_models(enabled_models, "../../trained_models", DEVICE, half, conf_path='../../dinov3')

f_cfg = FeatureConfig()

tr_cfg = TrainingConfig(feature_config=f_cfg, CRF=True, classifier='xgb', CRF_params={"label_confidence": 0.6,
    "sxy_g": [1, 1],
    "sxy_b": [30, 30],
    "s_rgb": [13, 13, 13],
    "compat_g": 10,
    "compat_b": 10,
    "n_infer": 10}, classifier_params={"class_weight": "balanced", "max_depth": 8}) 

Modifying Dv3 model w/ alibi attention


In [ ]:
ds_folder = 'data/trainable_seg'

AllowedImages = Literal['anode', 'curtains', 'bimodal', 'NMC', 'steel', 'rome', 'butterflies', 'au_foil']


side_lengths: dict[AllowedImages, int] = {
    'au_foil': 737,
    'anode': 756,
    'curtains': 756,
    'rome': 756,
    'butterflies': 756,
    'bimodal': 839,
    'NMC': 602,
    'steel': 602,
}


# image_fnames: list[AllowedImages] = ['rome', 'curtains', 'anode']
image_fnames: list[AllowedImages] = ['nmc_wide', 'bimodal', 'au_foil']
images: list[Image.Image] = []
labels: list[np.ndarray] = []
for image_fname in image_fnames:
    img = Image.open(f"{ds_folder}/{image_fname}.png").convert("RGB")
    shortest = min(img.size)
    L = side_lengths[image_fname]
    sf = L / shortest
    new_size = (int(img.size[0] * sf), int(img.size[1] * sf))
    img = img.resize(new_size, resample=Image.BILINEAR)
    images.append(img)

    label = load_labels(f"{ds_folder}/{image_fname}_biased_labels.tiff")
    labels.append(label)

FileNotFoundError: [Errno 2] No such file or directory: 'data/trainable_seg/NMC.png'

In [ ]:
features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}
red_features: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}


for key, model in models.items():
    for img in images:
        feats = get_features(model, img, False, False,  device=DEVICE)
        reduced = do_2D_pca(feats, 9, pre_norm='std', post_norm='minmax')
        reduced_hr = resize(reduced, (img.height, img.width), order=1, mode='reflect', anti_aliasing=True)

        classical = featurise(np.array(img.convert('L')), tr_cfg)

        combined_feats = concat_feats(reduced_hr, classical)
        features[key].append(combined_feats)

        red_features[key].append(reduced_hr)

2026-05-07 10:22:25 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1008) with `default`: weka-style features
2026-05-07 10:22:27 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1047) with `default`: weka-style features
2026-05-07 10:22:28 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1319) with `default`: weka-style features
2026-05-07 10:22:30 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1008) with `default`: weka-style features
2026-05-07 10:22:31 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1047) with `default`: weka-style features
2026-05-07 10:22:33 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1319) with `default`: weka-style features
2026-05-07 10:22:35 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1008) with `default`: weka-style features
2026-05-07 10:22:36 | I | multiscale_classical_cpu.py:426 | CPU feats on (756, 1047) with `default`: weka-style features
2026-05-07 10:22:38 | I | multis

In [ ]:
preds: dict[ModelTypes, list[np.ndarray]] = {k: [] for k in enabled_models}

for key, model in models.items():
    for i, feat in enumerate(features[key]):
        image = np.array(images[i])
        label = labels[i]

        pred, _, _ = train_and_apply(feat, label, tr_cfg, image=image)
        preds[key].append(pred)

2026-05-07 10:22:44 | I | core.py                    :122 | Training XGBClassifier: (12166, 67) -> ((12166,)) 


/home/ronan/miniconda3/envs/dv2/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:22:44] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-05-07 10:22:44 | I | core.py                    :137 | Applying XGBClassifier to (756, 1008, 67) features
2026-05-07 10:22:44 | I | crf.py                     :77 | CRF: probs (756, 1008, 4), img (756, 1008, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-05-07 10:22:45 | I | core.py                    :122 | Training XGBClassifier: (4182, 67) -> ((4182,)) 
2026-05-07 10:22:46 | I | core.py                    :137 | Applying XGBClassifier to (756, 1047, 67) features


/home/ronan/miniconda3/envs/dv2/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:22:45] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-05-07 10:22:46 | I | crf.py                     :77 | CRF: probs (756, 1047, 4), img (756, 1047, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-05-07 10:22:47 | I | core.py                    :122 | Training XGBClassifier: (2928, 67) -> ((2928,)) 
2026-05-07 10:22:47 | I | core.py                    :137 | Applying XGBClassifier to (756, 1319, 67) features
2026-05-07 10:22:47 | I | crf.py                     :77 | CRF: probs (756, 1319, 3), img (756, 1319, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


...

2026-05-07 10:22:56 | I | core.py                    :137 | Applying XGBClassifier to (756, 1008, 67) features
2026-05-07 10:22:56 | I | crf.py                     :77 | CRF: probs (756, 1008, 4), img (756, 1008, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-05-07 10:22:57 | I | core.py                    :122 | Training XGBClassifier: (4182, 67) -> ((4182,)) 
2026-05-07 10:22:58 | I | core.py                    :137 | Applying XGBClassifier to (756, 1047, 67) features


/home/ronan/miniconda3/envs/dv2/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:22:57] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-05-07 10:22:58 | I | crf.py                     :77 | CRF: probs (756, 1047, 4), img (756, 1047, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}
2026-05-07 10:22:59 | I | core.py                    :122 | Training XGBClassifier: (2928, 67) -> ((2928,)) 
2026-05-07 10:22:59 | I | core.py                    :137 | Applying XGBClassifier to (756, 1319, 67) features


/home/ronan/miniconda3/envs/dv2/lib/python3.12/site-packages/xgboost/training.py:199: UserWarning: [10:22:59] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-05-07 10:22:59 | I | crf.py                     :77 | CRF: probs (756, 1319, 3), img (756, 1319, 3), params: {
  "label_confidence": 0.6,
  "sxy_g": [
    1,
    1
  ],
  "sxy_b": [
    30,
    30
  ],
  "s_rgb": [
    13,
    13,
    13
  ],
  "compat_g": 10.0,
  "compat_b": 10.0,
  "n_infer": 10
}


In [ ]:
from skimage.color import label2rgb
from PIL.ImageColor import getcolor
from dinosaw.helpers import add_custom_font

COLOURS = [
    "#648FFF",
    "#785EF0",
    "#DC267F",
    "#FE6100",
    "#FFB000"
]
COLORS = [[v / 255.0 for v in getcolor(c, "RGB")] for c in COLOURS]



add_custom_font('resources/fonts', 'Grotesk')

def hide_axis(ax):
    ax.set_xticks([])
    ax.set_yticks([])



def apply_labels_as_overlay(labels: np.ndarray, img: Image.Image, colors: list, alpha: float=1.0) -> Image.Image:
    labels_unsqueezed = np.expand_dims(labels, -1)

    overlay = label2rgb(labels, colors=colors[1:], kind='overlay', bg_label=0, image_alpha=1, alpha=alpha)
    out = np.where(labels_unsqueezed, overlay * 255, np.array(img)).astype(np.uint8)
    img_with_labels = Image.fromarray(out)
    return img_with_labels

In [ ]:
%%capture
W, H = 3, 3
FS = 20
N_ROWS = len(enabled_models) + 1
N_COLS = len(image_fnames)
fig, axs = plt.subplots(N_ROWS, N_COLS, figsize=(4 * N_COLS, 2.8 * N_ROWS))

titles = ["Rome", "Cathode", "Anode"]

for y in range(N_ROWS):
    for x in range(N_COLS):
        if y == 0:
            axs[y, 0].set_ylabel("Image + labels", fontsize=FS)
            axs[y, x].set_title(titles[x], fontsize=FS)
            arr = apply_labels_as_overlay(labels[x], images[x], COLORS, alpha=1)
        else:
            key = enabled_models[y-1]
            model_name = model_names[key]
            model_name = model_name.replace('(COCO)', '')
            weight = 700 if 'alibi' in key.lower() else 500
            axs[y, 0].set_ylabel(model_name, fontsize=FS, weight=weight)
            arr = preds[key][x]
            arr = label2rgb(arr + 1, colors=COLORS[1:], bg_label=0)
        axs[y, x].imshow(arr, aspect='auto')
        hide_axis(axs[y, x])

plt.tight_layout()
plt.savefig('saved/06_dv3.jpeg', dpi=300, bbox_inches='tight', pil_kwargs={'optimize': True})